In [1]:
import os
import re
from sklearn import tree
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import seaborn as sns

pd.options.mode.chained_assignment

'warn'

In [7]:
df = pd.read_csv('/Users/ryan.oliveira/Desktop/puc/ppl-cd-pcd-sist-int-2025-1-grupo-5-2025-1/assets/data/State_of_data_BR_2023_original.csv')

In [15]:
valores_unicos_nivel = df[("('P2_g ', 'Nivel')")].unique()
print(valores_unicos_nivel)

['Sênior' 'Pleno' 'Júnior' nan]


In [24]:
nan_nivel = df[("('P2_g ', 'Nivel')")].isnull().sum()
print("Qtd 'nan' de nível: ", nan_nivel)

not_nan_count = df[("('P2_g ', 'Nivel')")].notnull().sum()
print("Qtd  de not 'nan' de nível: ", not_nan_count)

Qtd 'nan' de nível:  1436
Qtd  de not 'nan' de nível:  3857


In [26]:
nan_nivel = df[("('P2_h ', 'Faixa salarial')")].isnull().sum()
print("Qtd 'nan' de nível: ", nan_nivel)

not_nan_count = df[("('P2_h ', 'Faixa salarial')")].notnull().sum()
print("Qtd  de not 'nan' de nível: ", not_nan_count)

Qtd 'nan' de nível:  540
Qtd  de not 'nan' de nível:  4753


In [31]:
non_nan_count = df[("('P2_h ', 'Faixa salarial')")].notnull() & df[("('P2_g ', 'Nivel')")].notnull()

count_non_nan_rows = non_nan_count.sum()

print(f"Quantidade de linhas onde ambas as colunas não são nulas: {count_non_nan_rows}")

Quantidade de linhas onde ambas as colunas não são nulas: 3857


In [34]:
col_salario = "('P2_h ', 'Faixa salarial')"
col_nivel = "('P2_g ', 'Nivel')"

df_filtrado = df[df[col_salario].notnull() & df[col_nivel].notnull()]

df_nivel_salario = df_filtrado[[col_salario, col_nivel]]

print(df_nivel_salario.columns)

Index(['('P2_h ', 'Faixa salarial')', '('P2_g ', 'Nivel')'], dtype='object')


In [36]:
df_nivel_salario.rename(
    columns={
        "('P2_h ', 'Faixa salarial')": "faixa_salarial",
        "('P2_g ', 'Nivel')": "nivel",
    },
    inplace=False
)

print(df_nivel_salario.columns)

Index(['faixa_salarial', 'nivel'], dtype='object')


In [37]:
valores_unicos_salario = df_nivel_salario['faixa_salarial'].unique()
print(valores_unicos_salario)

['de R$ 12.001/mês a R$ 16.000/mês' 'de R$ 6.001/mês a R$ 8.000/mês'
 'de R$ 4.001/mês a R$ 6.000/mês' 'de R$ 8.001/mês a R$ 12.000/mês'
 'de R$ 1.001/mês a R$ 2.000/mês' 'de R$ 3.001/mês a R$ 4.000/mês'
 'de R$ 2.001/mês a R$ 3.000/mês' 'Acima de R$ 40.001/mês'
 'de R$ 16.001/mês a R$ 20.000/mês' 'de R$ 20.001/mês a R$ 25.000/mês'
 'Menos de R$ 1.000/mês' 'de R$ 25.001/mês a R$ 30.000/mês'
 'de R$ 30.001/mês a R$ 40.000/mês' 'de R$ 101/mês a R$ 2.000/mês']


In [39]:
valores_unicos_nivel = df_nivel_salario[('nivel')].unique()
print(valores_unicos_nivel)

['Sênior' 'Pleno' 'Júnior']


In [40]:
resultado = pd.crosstab(df_nivel_salario['faixa_salarial'], df_nivel_salario['nivel'])

print(resultado)

nivel                             Júnior  Pleno  Sênior
faixa_salarial                                         
Acima de R$ 40.001/mês                 0      2      26
Menos de R$ 1.000/mês                 22      3       0
de R$ 1.001/mês a R$ 2.000/mês       193     16       1
de R$ 101/mês a R$ 2.000/mês           1      0       0
de R$ 12.001/mês a R$ 16.000/mês       5     80     393
de R$ 16.001/mês a R$ 20.000/mês       3     20     139
de R$ 2.001/mês a R$ 3.000/mês       211     52      10
de R$ 20.001/mês a R$ 25.000/mês       0      7      73
de R$ 25.001/mês a R$ 30.000/mês       0      7      40
de R$ 3.001/mês a R$ 4.000/mês       195    126      12
de R$ 30.001/mês a R$ 40.000/mês       0      3      33
de R$ 4.001/mês a R$ 6.000/mês       294    356      53
de R$ 6.001/mês a R$ 8.000/mês        79    369     133
de R$ 8.001/mês a R$ 12.000/mês       43    351     506


## Tentativa de Balanceamento

- Menos de R$ 1.000/mês = Júnior
- de R$ 1.001/mês a R$ 2.000/mês = Júnior
- de R$ 101/mês a R$ 2.000/mês = Júnior
- de R$ 3.001/mês a R$ 4.000/mês = Júnior
- de R$ 2.001/mês a R$ 3.000/mê = Pleno
- de R$ 4.001/mês a R$ 6.000/mês = Pleno
- de R$ 6.001/mês a R$ 8.000/mês = Pleno
- de R$ 20.001/mês a R$ 25.000/mês = Sênior
- de R$ 25.001/mês a R$ 30.000/mês = Sênior
- de R$ 12.001/mês a R$ 16.000/mês = Sênior
- de R$ 16.001/mês a R$ 20.000/mês = Sênior
- de R$ 30.001/mês a R$ 40.000/mês = Sênior
- de R$ 8.001/mês a R$ 12.000/mês = Sênior
- Acima de R$ 40.001/mês = Sênior



```python
Melhores hiperparâmetros: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Acurácia no conjunto de teste: 0.47432503970354684
Acurácia no conjunto de treinamento: 0.519140362659503
```